<a href="https://colab.research.google.com/github/arina674410-boop/pfo-af-meta-analysis/blob/main/hip_fracture_clinical_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## import libraries

In [ ]:
!pip install pandas openpyxl



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
!pip install matplotlib


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


# Exploratory Data Analysis of Hospitalizations for Hip and Femoral Fractures
##  Study Objective
The objective of this analysis is to identify factors associated with clinical outcomes, length of hospital stay, complications, readmissions, and mortality among patients hospitalized with hip and femoral fractures. The study also evaluates the completeness and quality of healthcare delivery across participating institutions.

## Analytical Workflow
Data loading and initial inspection
Data cleaning and preprocessing
Descriptive statistics and cohort characterization
Analysis of surgical treatment patterns and patient pathways
Analysis of medication administration
Investigation of microbiological testing and infection-related factors
Evaluation of clinical outcomes and 30-day readmissions
Correlation and multivariate analysis
Generation of hypotheses and key findings


# Dataset Description

The dataset contains anonymized hospitalization records of patients admitted with hip and femoral fractures.

Key variable groups include:

- Demographic characteristics
- Admission and discharge information
- Surgical procedures
- Medication administration
- Microbiological investigations
- Clinical outcomes
- Mortality and readmissions

### required packages

In [ ]:
!pip install seaborn


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
# libraries
import pandas as pd
import numpy as np

# visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# clear
import warnings
warnings.filterwarnings('ignore')

# styles
sns.set_style('whitegrid')
sns.set_palette('pastel')
%matplotlib inline

In [ ]:
!pip install phik


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from phik import phik_matrix

## initial data inspection

In [ ]:
# load data
df = pd.read_excel('filtered_bedr_data_ex.xlsx', engine='openpyxl')

# check
df.head()

In [ ]:
# count
print(f"size: {df.shape[0]} rows and {df.shape[1]} columns\n")

# presence NaN
df.info()

## cohort characteristics

In [ ]:
# 1. cohort characteristics

# total hosp
total_hosp = df.shape[0]
# count unique patients
unique_patients = df['erz_number'].nunique()
# unique facilities
unique_facilities = df['facility_id'].nunique()

print(f"total amount of hospit: {total_hosp}")
print(f"amount of patients: {unique_patients}")
print(f"amount of facilities: {unique_facilities}")

# multihosp
multi_hosp_patients = df['erz_number'].value_counts()
multi_hosp_patients = multi_hosp_patients[multi_hosp_patients > 1]
print(f"amount of patients who has multihosp {len(multi_hosp_patients)}")

In [ ]:
# age structure
print("age structure")
display(df['age'].describe().round(1))

# visualisation
plt.figure(figsize=(10, 5))
# bins 5 years
sns.histplot(df['age'], bins=range(0, 101, 5), kde=True, color='skyblue', edgecolor='black')
plt.title('age structure', fontsize=14)
plt.xlabel('age', fontsize=12)
plt.ylabel('number of patients', fontsize=12)
plt.xticks(range(0, 101, 10))
plt.grid(axis='y', alpha=0.75)
plt.show()

In [ ]:
# sex structure
sex_counts = df['sex'].value_counts()
sex_percents = df['sex'].value_counts(normalize=True) * 100

print("sex structure")
display(pd.concat([sex_counts, sex_percents], axis=1, keys=['num', '%']).round(2))

# visualisation
plt.figure(figsize=(6, 4))
ax = sns.countplot(data=df, x='sex', palette='Set2', edgecolor='black')
plt.title('sex structure', fontsize=14)
plt.xlabel('sex', fontsize=12)
plt.ylabel('number', fontsize=12)

# percents
for p in ax.patches:
    ax.annotate(f'{p.get_height() / len(df) * 100:.1f}%',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=11, fontweight='bold')
plt.grid(axis='y', alpha=0.75)
plt.show()

## hospital-level analysis: top 10 facilities by case volume

In [ ]:
# pivot table
facility_stats = df.groupby('lpu_name').agg(
    total_hosp=('hospitalization_id', 'count'),
    unique_patients=('erz_number', 'nunique'),
    avg_age=('age', 'mean'),
    male_ratio=('sex', lambda x: (x == 'М').mean() * 100), # man в %
    death_in_hosp_ratio=('death_in_hospitalization', lambda x: x.mean() * 100) # mortality %
).round(2).sort_values(by='total_hosp', ascending=False)

print("top 10 facilities be case volume):")
display(facility_stats.head(10))

# visualisation
top_10_facilities = facility_stats.head(10).index

plt.figure(figsize=(12, 6))
sns.barplot(
    data=df[df['lpu_name'].isin(top_10_facilities)],
    x='lpu_name',
    y='death_in_hospitalization',
    estimator=lambda x: x.mean() * 100, # average percent (flag 1 or 0)
    errorbar=None,
    palette='Reds_r',
    edgecolor='black'
)
plt.title('mortality. top 10 facilities', fontsize=14)
plt.xlabel('name', fontsize=12)
plt.ylabel('mortality %', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.75)

# average line
overall_death_rate = df['death_in_hospitalization'].mean() * 100
plt.axhline(overall_death_rate, color='red', linestyle='--', linewidth=2,
            label=f'average: {overall_death_rate:.2f}%')
plt.legend()
plt.tight_layout()
plt.show()

## outlier detection and time-to-surgery analysis.
### this section examines distributions and potential outliers in patient age, length of stay, and time to surgery.

In [ ]:
# 1. age - elderly\young
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# hystogr
axes[0].hist(df['age'], bins=range(0, 110, 5), edgecolor='black', alpha=0.7)
axes[0].set_xlabel('age')
axes[0].set_ylabel('number of patients')
axes[0].set_title('age structure')
axes[0].grid(axis='y', alpha=0.75)

# age boxplot
axes[1].boxplot(df['age'].dropna(), vert=True)
axes[1].set_ylabel('age')
axes[1].set_title('age boxplot')
axes[1].grid(axis='y', alpha=0.75)

plt.tight_layout()
plt.show()

# stats
print("\nage structure")
print(df['age'].describe())
print(f"\nmin age {df['age'].min()}")
print(f"max age: {df['age'].max()}")
print(f"\n<18")
print(df[df['age'] < 18][['hospitalization_id', 'age', 'sex']].head(10))
print(f"\n>100:")
print(df[df['age'] > 100][['hospitalization_id', 'age', 'sex']].head(10))

In [ ]:
# 2. hosp duration
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# hystigr
axes[0].hist(df['bed_days'], bins=range(0, 100, 2), edgecolor='black', alpha=0.7)
axes[0].set_xlabel('days')
axes[0].set_ylabel('number of patients')
axes[0].set_title('structure of hospital stay duration')
axes[0].grid(axis='y', alpha=0.75)
axes[0].set_xlim(0, 60)  # restrict X

# Boxplot of dur
axes[1].boxplot(df['bed_days'].dropna(), vert=True)
axes[1].set_ylabel('days')
axes[1].set_title('duration boxplot')
axes[1].grid(axis='y', alpha=0.75)

plt.tight_layout()
plt.show()

# stat
print("\nstructure of hospital stay duration:")
print(df['bed_days'].describe())
print(f"\ntop 10:")
print(df.nlargest(10, 'bed_days')[['hospitalization_id', 'bed_days', 'input_date', 'leave_date']])

In [ ]:
#convertion of columns
date_cols = [
    'input_date', 'leave_date',
    'first_operation_start_dt', 'first_operation_end_dt',
    'death_date_in_hospitalization', 'death_date_within_30_days'
]

for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

print("dates converted")
for col in date_cols:
    if col in df.columns:
        print(f"  {col}: {df[col].dtype}")

In [ ]:
# hours_to_operation
df['hours_to_operation'] = (df['first_operation_start_dt'] - df['input_date']).dt.total_seconds() / 3600

# visual
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# hystogr
valid_hours = df['hours_to_operation'].dropna()
valid_hours_clipped = valid_hours[valid_hours <= 100]  # minus extrim

axes[0].hist(valid_hours_clipped, bins=50, edgecolor='black', alpha=0.7, color='coral')
axes[0].set_xlabel('hours to operation')
axes[0].set_ylabel('number of patients')
axes[0].set_title('structure of delay')
axes[0].grid(axis='y', alpha=0.75)

# lines
axes[0].axvline(24, color='green', linestyle='--', linewidth=2, label='24 часа (целевой срок)')
axes[0].axvline(48, color='orange', linestyle='--', linewidth=2, label='48 часа')
axes[0].legend()

# Boxplot
axes[1].boxplot(valid_hours[valid_hours <= 100].dropna(), vert=True,
                patch_artist=True,  # add this par
                boxprops=dict(facecolor='lightcoral', color='darkred'),
                medianprops=dict(color='darkred', linewidth=2))
axes[1].set_ylabel('hours to operation')
axes[1].set_title('Boxplot')
axes[1].grid(axis='y', alpha=0.75)

plt.tight_layout()
plt.show()

# stat
print("\nstructure of delay")
print(valid_hours.describe())

# percentage of surgical interventions within target timeframes
total_ops = valid_hours.count()
within_24h = (valid_hours <= 24).sum()
within_48h = (valid_hours <= 48).sum()
over_48h = (valid_hours > 48).sum()

print(f"\n⏱percentage of surgical interventions within target timeframes:")
print(f"  ≤ 24 h: {within_24h} ({within_24h/total_ops*100:.1f}%)")
print(f"  ≤ 48 h: {within_48h} ({within_48h/total_ops*100:.1f}%)")
print(f"  > 48 h: {over_48h} ({over_48h/total_ops*100:.1f}%)")

# top 10 of delays
print(f"\ntop 10:")
top_delays = df.nlargest(10, 'hours_to_operation')[[
    'hospitalization_id', 'erz_number', 'age', 'sex',
    'input_date', 'first_operation_start_dt', 'hours_to_operation',
    'lpu_name'
]]
display(top_delays)

# Advanced Analysis

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. mark of oper
df['has_operation'] = ((df['operation_osteo_flag'] == 1) |
                       (df['operation_endoprotez_flag'] == 1) |
                       (df['other_operation_flag'] == 1)).astype(int)

# 2. hous to operation
df['hours_to_operation'] = (df['first_operation_start_dt'] - df['input_date']).dt.total_seconds() / 3600
# exclude incorrect values
df.loc[df['hours_to_operation'] < 0, 'hours_to_operation'] = np.nan

# 3. duration of the period before death
df['days_to_death'] = (df['death_date_in_hospitalization'] - df['input_date']).dt.days

# 4. fact of re-hospitalization
df['has_repeat_hosp'] = df['repeat_hosp_30d_info'].notna().astype(int)

print("done")

In [ ]:
def calc_facility_metrics(group):
    """function for calculating all metrics for one fac"""
    res = {}
    res['number of hosp'] = len(group)

    # surg act
    res['oper %'] = group['has_operation'].mean() * 100

    # timeliness
    valid_hours = group['hours_to_operation'].dropna()
    res['median, h'] = valid_hours.median() if len(valid_hours) > 0 else np.nan
    res['≤24, %'] = (valid_hours <= 24).mean() * 100 if len(valid_hours) > 0 else np.nan
    res['≤48, %'] = (valid_hours <= 48).mean() * 100 if len(valid_hours) > 0 else np.nan
    res['>48ч, %'] = (valid_hours > 48).mean() * 100 if len(valid_hours) > 0 else np.nan

    # stay dur
    res['av duration d'] = group['bed_days'].mean()
    res['med duration d'] = group['bed_days'].median()

    # mortality
    res['death in hospitalisation, %'] = group['death_in_hospitalization'].mean() * 100
    res['death within 30 days, %'] = group['death_within_30_days'].mean() * 100

    # time of death (only for those who died in hospital)
    valid_death_days = group['days_to_death'].dropna()
    res['Медиана сроков смерти, дн'] = valid_death_days.median() if len(valid_death_days) > 0 else np.nan

    # re-hosp
    res['re-hosp (30 days), %'] = group['has_repeat_hosp'].mean() * 100

    return pd.Series(res)

# all facilities
stats = df.groupby('lpu_name').apply(calc_facility_metrics).reset_index()

# fill in the blanks with zeros where it makes sense (the median time of death if there were no deaths)
stats = stats.fillna(0)
print(f" done for {len(stats)} facilities.")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# paramet vis
sns.set_style('whitegrid')
sns.set_palette('muted')
plt.rcParams['figure.figsize'] = (14, 8)

# garantee of dates
date_cols = ['input_date', 'first_operation_start_dt', 'death_date_in_hospitalization']
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

# create der features
# operation yes no
df['has_operation'] = ((df['operation_osteo_flag'] == 1) |
                       (df['operation_endoprotez_flag'] == 1) |
                       (df['other_operation_flag'] == 1)).astype(int)

# hours to hosp
df['hours_to_operation'] = (df['first_operation_start_dt'] - df['input_date']).dt.total_seconds() / 3600
# ex of incorrec
df.loc[df['hours_to_operation'] < 0, 'hours_to_operation'] = np.nan

# timing of death
df['days_to_death'] = (df['death_date_in_hospitalization'] - df['input_date']).dt.days

# rehosp yes or no
df['has_readmission'] = df['repeat_hosp_30d_info'].notna().astype(int)

print("done")

In [ ]:
# basic indicators
total_hosp = len(df)
total_patients = df['erz_number'].nunique()

# age and sex
age_mean = df['age'].mean()
age_median = df['age'].median()
age_min, age_max = df['age'].min(), df['age'].max()
sex_counts = df['sex'].value_counts()
sex_pcts = df['sex'].value_counts(normalize=True) * 100

# surgical activity
op_rate = df['has_operation'].mean() * 100
osteo_rate = df['operation_osteo_flag'].mean() * 100
endo_rate = df['operation_endoprotez_flag'].mean() * 100
other_op_rate = df['other_operation_flag'].mean() * 100

# timeliness
valid_hours = df['hours_to_operation'].dropna()
median_hours = valid_hours.median()
pct_le_24 = (valid_hours <= 24).mean() * 100
pct_le_48 = (valid_hours <= 48).mean() * 100
pct_gt_48 = (valid_hours > 48).mean() * 100

# duration
mean_los = df['bed_days'].mean()
median_los = df['bed_days'].median()

# mostality
hosp_mort_rate = df['death_in_hospitalization'].mean() * 100
mort_30d_rate = df['death_within_30_days'].mean() * 100

dead_in_hosp = df[df['death_in_hospitalization'] == 1]
valid_death_days = dead_in_hosp['days_to_death'].dropna()
median_days_to_death = valid_death_days.median() if not valid_death_days.empty else np.nan

# rehospitalisations
readmission_rate = df['has_readmission'].mean() * 100

print(" done")

In [ ]:
fig = plt.figure(figsize=(18, 12))
fig.suptitle('pooled analysis of a cohort of patients with hip/femoral neck fractures (all facilities)',
             fontsize=18, fontweight='bold', y=0.98)

# 1. age structure (Pie)
ax1 = plt.subplot(2, 3, 1)
sex_counts.plot(kind='pie', autopct='%1.1f%%', ax=ax1, colors=['#ff9999','#66b3ff'], startangle=90)
ax1.set_ylabel('')
ax1.set_title('age structure')

# 2. age str (Hist + KDE)
ax2 = plt.subplot(2, 3, 2)
sns.histplot(df['age'], bins=30, kde=True, color='purple', ax=ax2)
ax2.axvline(age_median, color='red', linestyle='--', label=f'median: {age_median:.0f} y.o')
ax2.set_title(f'age srtucture\n(mean: {age_mean:.1f}, min-max: {age_min}-{age_max})')
ax2.set_xlabel('age')
ax2.legend()

# 3. operations (Bar)
ax3 = plt.subplot(2, 3, 3)
op_struct = {
    'osteosyntesis': osteo_rate,
    'endoprosthetics': endo_rate,
    'other': other_op_rate,
    'without oper': 100 - op_rate
}
sns.barplot(x=list(op_struct.keys()), y=list(op_struct.values()), palette='viridis', ax=ax3)
ax3.set_title('structure of surg activity (%)')
ax3.set_ylabel('%')
ax3.tick_params(axis='x', rotation=15)

# 4. time to operation (Hist)
ax4 = plt.subplot(2, 3, 4)
# trim the tail of the distribution for clarity (up to 120 hours)
sns.histplot(valid_hours[valid_hours <= 120], bins=40, kde=True, color='teal', ax=ax4)
ax4.axvline(24, color='green', linestyle='--', linewidth=2, label='target time 24')
ax4.axvline(48, color='orange', linestyle='--', linewidth=2, label='critical time 48')
ax4.set_title(f'time: entrance -> operation"\n(median: {median_hours:.1f} ч, N={len(valid_hours)})')
ax4.set_xlabel('hours')
ax4.legend()

# 5. duration (Boxplot)
ax5 = plt.subplot(2, 3, 5)
# trim the tail
clip_val = df['bed_days'].quantile(0.95)
sns.boxplot(y=df['bed_days'][df['bed_days'] <= clip_val], ax=ax5, color='lightgreen')
ax5.set_title(f'duration\n(median: {median_los:.0f} дн, mean: {mean_los:.1f} дн)')
ax5.set_ylabel('days')

# 6. mortality and rehosp (Bar)
ax6 = plt.subplot(2, 3, 6)
outcomes = {
    'mortalitu in hospitalisation': hosp_mort_rate,
    '30-d mort': mort_30d_rate,
    'rehosp 30 days': readmission_rate
}
sns.barplot(x=list(outcomes.keys()), y=list(outcomes.values()), palette=['#e74c3c', '#c0392b', '#e67e22'], ax=ax6)
ax6.set_title('adverse outcomes (%)')
ax6.set_ylabel('%')
ax6.tick_params(axis='x', rotation=15)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

In [ ]:
report = f"""
SUMMARY ANALYSIS OF A PATIENT COHORT
01.01.2025 – 31.03.2026

1. GENERAL CHARACTERISTICS AND DEMOGRAPHICS
• Total number of hospitalizations:{total_hosp} (unique patients: {total_patients}).
• Age structure: The average age of patients is {age_mean:.1f} y.o (median: {age_median:.0f} y.o).
  Age range: from {age_min} to {age_max} y.o.
• Sex structure: Women - {sex_pcts.get('Ж', 0):.1f}% ({sex_counts.get('Ж', 0)} ), мужчины — {sex_pcts.get('М', 0):.1f}% ({sex_counts.get('М', 0)} ).

2. SURGICAL ACTIVITY AND TIME
• Percentage of patients operated on:{op_rate:.1f}% (total number of operations performed: {df['has_operation'].sum()}).
• Structure of interventions:
- Osteosynthesis: {osteo_rate:.1f}%
  - Endoprosthetics: {endo_rate:.1f}%
  - other: {other_op_rate:.1f}%
• Timeliness of operations (of all patients operated on):
- Median time from admission to surgery: {median_hours:.1f} h.
  - Percentage of operations completed ≤ 24 hours:{pct_le_24:.1f}%
  - Percentage of surgeries completed ≤ 48 hours: {pct_le_48:.1f}%
  - Percentage of surgeries with delays > 48 hours: {pct_gt_48:.1f}%

3. DURATION OF TREATMENT AND MORTALITY RATE
• Average length of hospital stay: {mean_los:.1f} days (median: {median_los:.0f} d).
• Hospital mortality: {hosp_mort_rate:.2f}%.
• 30-day mortality: {mort_30d_rate:.2f}%.
• Time to death (in hospital): The median is {median_days_to_death:.0f} days.

4. POSTHOSPITAL EVENTS
• Readmission rate within 30 days of discharge: {readmission_rate:.1f}%.
"""

print(report)